# Supplement — study regions + 25 May 2026 overview (Levante)

One map, on the ERA5 fields, showing:

* **25 May 2026 daily-mean T2m anomaly** (0.5°, vs 1995–2025 ±3-day climatology) — shading
* **Z500 geopotential-height contours** (1°, 25 May 2026) — black
* **All regions used in the study** (as boxes):
  * Target region (UK–France)
  * Analogue selection region (`BASE_BOX`)
  * Moroccan branch box
  * Cut-off-low selection box (large classification box)
  * Extended cut-off-low **preselection** box (colindex2 detection region)
  * **IFS trajectory domain** — drawn as **5° corner stubs only** (the box is far larger than the map)
  * European RMST box
* the **detected 2026 cut-off** (colindex2, same detector as the catalogue) with its **centre + radius circle**

Run on Levante in the kernel that has `colindex2` + `cartopy` (the analogue/catalogue kernel).
Paths are taken verbatim from `paper_plot_v4-2.ipynb` and `ANALOGUES_timing_anomaly_from_catalogue_version3-5.ipynb`.

In [ ]:
import os, glob
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import matplotlib as mpl, matplotlib.pyplot as plt, matplotlib.colors as mcolors
import cartopy.crs as ccrs, cartopy.feature as cfeature
import cartopy.geodesic as cgeo
mpl.rcParams["figure.dpi"] = 150

# ---------------- data paths (Levante; from the paper-plot / timing notebooks) ----------------
path_project = "${ANALOGUE_BASE}/"
T2M_0P5_FILE = os.path.join(path_project, "rawdat_0p5", "era5_t2m_daily_NA_1940_2026_FULL_0p5deg.nc")  # panel-a shading
Z500_FILE    = os.path.join(path_project, "rawdat",     "era5_z500_daily_NA_1940_2026_FULL_1deg.nc")   # contours + detector

# ---------------- event + climatology (same as paper_plot_v4-2) ----------------
TARGET_DAY = "2026-05-25"
CLIM_START_YEAR, CLIM_END_YEAR, CLIM_HALF_WINDOW_DAYS = 1995, 2025, 3
G0 = 9.80665

# ---------------- regions drawn as full boxes  (lon0, lon1, lat0, lat1) ----------------
# NOTE: edit the Moroccan branch box here if your definition differs (study value = -20..0, 25..40).
# green = analogue selection + RMST (shared);  black dashed = target;  black dotted = Moroccan branch;
# orange = cut-off selection (distinct from the blue-red anomaly colormap; no brown).
GREEN = "#2ca02c"
BOXES = {
    "Target region (UK-France)":     dict(box=(-5.5,  5.0, 44.5, 55.0), color="black",   ls="-", lw=3),
    "Analogue selection":     dict(box=(-20.0, 25.0, 30.0, 68.0), color="midnightblue",    ls="-",  lw=2),
    "Moroccan branch":           dict(box=(-20.0,  0.0, 25.0, 40.0), color="black",   ls=":",  lw=3),
    "Cut-off selection": dict(box=(-35.0,  0.0, 25.0, 50.0), color="black", ls="-.",  lw=3),
    "European RMST":             dict(box=(-15.0, 30.0, 35.0, 70.0), color="midnightblue",    ls=":", lw=2),
}

# Cut-off PRESELECTION (colindex2 detection region, -50..55 / 5..90): far larger than the map,
# so drawn as 20-deg edge stubs pointing OUTWARD from the cut-off selection box corners.
SEL_BOX        = (-35.0, 0.0, 25.0, 50.0)    # = cut-off selection box (stubs attach here)
PRESEL_EDGE_DEG = 20.0

T_ANOM_LIMIT = 12                            # colour range for the T2m anomaly [degC]
print("event:", TARGET_DAY)

In [ ]:
# ---------------- IPCC temp_div colormap (local -> Levante copy -> RdBu_r) ----------------
def load_temp_div(fallback="RdBu_r"):
    for p in ["temp_div.txt",
              "${HOME}/colormaps_IPCC/continuous_colormaps_rgb_0-1/temp_div.txt"]:
        if os.path.exists(p):
            return mcolors.LinearSegmentedColormap.from_list("temp_div", np.loadtxt(p))
    return plt.get_cmap(fallback)
TEMPDIV = load_temp_div()

def to_pm180(da):
    ln = "lon" if "lon" in da.coords else "longitude"
    if float(da[ln].max()) > 180:
        da = da.assign_coords({ln: (((da[ln] + 180) % 360) - 180)}).sortby(ln)
    return da, ln, ("lat" if "lat" in da.coords else "latitude")

In [ ]:
# ---------------- 25 May 2026 T2m anomaly (0.5 deg, degC) ----------------
da = xr.open_dataarray(T2M_0P5_FILE).squeeze(drop=True)
if float(da.isel(time=0).mean()) > 100:          # K -> degC
    da = da - 273.15
t   = pd.DatetimeIndex(da["time"].values)
tgt = pd.Timestamp(TARGET_DAY).dayofyear
dist = np.abs(((t.dayofyear.values - tgt + 182) % 365) - 182)
win  = (dist <= CLIM_HALF_WINDOW_DAYS) & (t.year >= CLIM_START_YEAR) & (t.year <= CLIM_END_YEAR)
clim = da.isel(time=np.where(win)[0]).mean("time")
anom = (da.sel(time=pd.Timestamp(TARGET_DAY), method="nearest") - clim).load()
anom, LONN, LATN = to_pm180(anom)
print(f"T2m clim from {int(win.sum())} days; anomaly range {float(anom.min()):+.1f} .. {float(anom.max()):+.1f} degC")

# ---------------- 25 May 2026 Z500 height (1 deg, m) ----------------
z = xr.open_dataarray(Z500_FILE).squeeze(drop=True).sel(time=pd.Timestamp(TARGET_DAY), method="nearest").load()
z, ZLON, ZLAT = to_pm180(z)
ZH = z / G0
print("Z500 height range:", float(ZH.min()), "..", float(ZH.max()), "m")

In [ ]:
# ---------------- detect the 2026 cut-off (colindex2, identical detector to the catalogue) ----------------
# colindex2 fails to set coords on a SINGLE time step, so — exactly like the timing notebook —
# we detect over a short multi-day window and then read the TARGET day's feature CSV.
from colindex2 import Detect
DET_LON, DET_LAT = (-50, 55), (5, 90)              # = preselection / detection region
RADII            = np.arange(200, 2101, 100)
SO_DETECT, SO_CUTOFF, SR_MAX = 3.0, 6.0, 3.0
CLS_BOX = (-35.0, 0.0, 25.0, 50.0)                 # large classification box (where the event cut-off lives)

ts  = pd.Timestamp(TARGET_DAY)
WIN = pd.date_range(ts - pd.Timedelta(days=4), ts, freq="D")   # multi-day -> colindex2 sets coords

def _load_z_det(days):
    src = xr.open_dataarray(Z500_FILE); h = src.sel(time=pd.DatetimeIndex(days)).load(); src.close()
    for d in list(h.dims):
        if d not in ("time", "lat", "lon"):
            h = h.isel({d: 0}, drop=True)
    h = h.sel(lon=slice(*DET_LON))
    h = h.sel(lat=slice(DET_LAT[1], DET_LAT[0])) if float(h.lat[0]) > float(h.lat[-1]) else h.sel(lat=slice(*DET_LAT))
    return (h / G0).assign_attrs(units="m", standard_name="geopotential_height").assign_coords(level=500).transpose("time", "lat", "lon")

outdir = Path(f"./colindex_event_{ts:%Y%m%d}"); outdir.mkdir(parents=True, exist_ok=True)
prev = os.getcwd(); os.chdir(outdir)
try:
    Detect(_load_z_det(WIN), odir="out", ty="L", r=RADII, SR_thres=SR_MAX, So_thres=SO_DETECT,
           Do_thres=0.0, xx_thres=0.0, rm_rmin=True, rm_rmax=True, local_ex_req_num=7,
           gpu=False, chunk_size=100_000, nc=True, subs=False)
finally:
    os.chdir(prev)

csv = outdir / "out" / "V" / ts.strftime("%Y%m") / ("V-L-" + ts.strftime("%Y%m%d%H%M") + "-0500.csv")
pts = pd.read_csv(csv)
RADCOL = next((c for c in ("ro", "radius_km", "r") if c in pts.columns), None)
pts["lon180"] = (pts["lon"] + 180) % 360 - 180
pts["ex_i"]   = pd.to_numeric(pts["ex"], errors="coerce").fillna(0).astype(int)

cut = pts[(pts.ex_i == 1) & (pts.So >= SO_CUTOFF)
          & pts.lon180.between(CLS_BOX[0], CLS_BOX[1]) & pts.lat.between(CLS_BOX[2], CLS_BOX[3])]
if cut.empty:                                       # fallback: strongest closed cut-off anywhere in the detection region
    print("[WARN] no cut-off in the large box; falling back to the strongest ex==1 in the detection region")
    cut = pts[(pts.ex_i == 1) & (pts.So >= SO_CUTOFF)]
cut = cut.sort_values("So", ascending=False)
print(cut[["lon180", "lat", "So", "ex_i"] + ([RADCOL] if RADCOL else [])].head().to_string(index=False))

c = cut.iloc[0]
COL_LON, COL_LAT = float(c["lon180"]), float(c["lat"])
COL_R = float(c[RADCOL]) if RADCOL else 500.0       # km (colindex2 detection radius)
print(f"\ndetected cut-off: lon={COL_LON:.1f}, lat={COL_LAT:.1f}, So={c['So']:.1f}, radius={COL_R:.0f} km")

In [ ]:
# ---------------- the figure ----------------
proj = ccrs.PlateCarree()
# map extent = the plotted-data (T2m anomaly) extent
MAP_EXTENT = [float(anom[LONN].min()), float(anom[LONN].max()),
              float(anom[LATN].min()), float(anom[LATN].max())]
print("map extent (data):", [round(v, 1) for v in MAP_EXTENT])

fig = plt.figure(figsize=(13, 10)); ax = fig.add_subplot(111, projection=proj)
ax.set_extent(MAP_EXTENT, crs=proj)
ax.add_feature(cfeature.LAND, alpha=0.30); ax.add_feature(cfeature.OCEAN, alpha=0.15)
ax.coastlines("10m", linewidth=0.5); ax.add_feature(cfeature.BORDERS, linewidth=0.3)

# T2m anomaly shading (same colormap / TwoSlopeNorm as paper_plot panel a)
norm = mcolors.TwoSlopeNorm(vmin=-T_ANOM_LIMIT, vcenter=0, vmax=T_ANOM_LIMIT)
mesh = ax.pcolormesh(anom[LONN], anom[LATN], anom, cmap=TEMPDIV, norm=norm, shading="auto", transform=proj)
cb = fig.colorbar(mesh, ax=ax, orientation="vertical", shrink=0.70, pad=0.02)
cb.set_label(f"ERA5 daily-mean T2m anomaly [°C]  ({TARGET_DAY})", fontsize=12)

# Z500 height contours
cs = ax.contour(ZH[ZLON], ZH[ZLAT], ZH, levels=np.arange(4800, 6000, 40),
                colors="black", linewidths=0.7, transform=proj)
ax.clabel(cs, inline=True, fontsize=7, fmt="%d")

# study-region boxes
def draw_box(b, color, ls, lw, label):
    lo0, lo1, la0, la1 = b
    ax.plot([lo0, lo1, lo1, lo0, lo0], [la0, la0, la1, la1, la0],
            color=color, ls=ls, lw=lw, transform=proj, label=label, zorder=6)
for name, cfg in BOXES.items():
    draw_box(cfg["box"], cfg["color"], cfg["ls"], cfg["lw"], name)

# Cut-off preselection: 20-deg edge stubs pointing OUTWARD from the selection-box corners
lo0, lo1, la0, la1 = SEL_BOX; e = PRESEL_EDGE_DEG; pcol = "black"   # match cut-off selection box
outward = [(lo0, la0, -1, -1), (lo1, la0, +1, -1), (lo1, la1, +1, +1), (lo0, la1, -1, +1)]
for i, (x, y, sx, sy) in enumerate(outward):
    ax.plot([x, x + sx * e], [y, y], color=pcol, ls=":", lw=1.8, transform=proj, zorder=6,
            label=f"Cut-off preselection with {e:.0f}° edge" if i == 0 else None)
    ax.plot([x, x], [y, y + sy * e], color=pcol, ls=":", lw=1.8, transform=proj, zorder=6)

# detected cut-off: centre + geodesic radius circle
circle = np.asarray(cgeo.Geodesic().circle(lon=COL_LON, lat=COL_LAT, radius=COL_R * 1000.0, n_samples=200))
ax.plot(circle[:, 0], circle[:, 1], color="magenta", lw=1, transform=proj, zorder=8,
        label=f"Detected cut-off (r={COL_R:.0f} km)")
ax.plot(COL_LON, COL_LAT, marker="*", color="magenta", markersize=8, markeredgewidth=2,
        transform=proj, zorder=9)

gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.4)
gl.top_labels = gl.right_labels = False
ax.legend(loc="lower left", fontsize=8, framealpha=0.92, ncol=2)

fig.savefig("paper_figures/supplement_study_regions_2026-05-25.pdf", bbox_inches="tight", dpi=200)
fig.savefig("paper_figures/supplement_study_regions_2026-05-25.png", bbox_inches="tight", dpi=200)
plt.show()
print("saved supplement_study_regions_2026-05-25.pdf / .png")